# Decision Tree Weather Type Classification

This notebook builds one model only: a **Decision Tree classifier** for the Kaggle Weather Type Classification dataset.  
It follows the same workflow as the Random Forest version so the group project remains consistent across team members.

The notebook includes:
- dataset loading
- feature/target split
- preprocessing with a scikit-learn pipeline
- Decision Tree training
- model evaluation
- cross-validation
- feature importance analysis
- final testing with new weather conditions


## 1. Imports

This cell imports the libraries used for data loading, preprocessing, model building, evaluation, and plotting.  
These tools are enough to take the dataset from raw CSV form all the way to final predictions.

In [20]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

## 2. Load Dataset

This cell reads the weather dataset from a CSV file.  
If the file name is correct and the CSV is in the same folder as the notebook, the dataset will load into a pandas DataFrame.

In [21]:
DATA_PATH = "weather_classification_data.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,Temperature,Humidity,Wind Speed,Precipitation (%),Cloud Cover,Atmospheric Pressure,UV Index,Season,Visibility (km),Location,Weather Type
0,14.0,73,9.5,82.0,partly cloudy,1010.82,2,Winter,3.5,inland,Rainy
1,39.0,96,8.5,71.0,partly cloudy,1011.43,7,Spring,10.0,inland,Cloudy
2,30.0,64,7.0,16.0,clear,1018.72,5,Spring,5.5,mountain,Sunny
3,38.0,83,1.5,82.0,clear,1026.25,7,Spring,1.0,coastal,Sunny
4,27.0,74,17.0,66.0,overcast,990.67,1,Winter,2.5,mountain,Rainy


## 3. Inspect the dataset structure

This cell checks the shape of the dataset, the data types of each column, and summary statistics.  
This helps us understand what kind of preprocessing is needed before training the model.

In [22]:
print("Dataset shape:", df.shape)
display(df.info())
df.describe(include="all")

Dataset shape: (13200, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13200 entries, 0 to 13199
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Temperature           13200 non-null  float64
 1   Humidity              13200 non-null  int64  
 2   Wind Speed            13200 non-null  float64
 3   Precipitation (%)     13200 non-null  float64
 4   Cloud Cover           13200 non-null  object 
 5   Atmospheric Pressure  13200 non-null  float64
 6   UV Index              13200 non-null  int64  
 7   Season                13200 non-null  object 
 8   Visibility (km)       13200 non-null  float64
 9   Location              13200 non-null  object 
 10  Weather Type          13200 non-null  object 
dtypes: float64(5), int64(2), object(4)
memory usage: 1.1+ MB


None

,Temperature,Humidity,Wind Speed,Precipitation (%),Cloud Cover,Atmospheric Pressure,UV Index,Season,Visibility (km),Location,Weather Type
count,13200.000000,13200.000000,13200.000000,13200.000000,13200,13200.000000,13200.000000,13200,13200.000000,13200,13200
unique,NaN,NaN,NaN,NaN,4,NaN,NaN,4,NaN,3,4
top,NaN,NaN,NaN,NaN,overcast,NaN,NaN,Winter,NaN,inland,Rainy
freq,NaN,NaN,NaN,NaN,6090,NaN,NaN,5610,NaN,4816,3300
mean,19.127576,68.710833,9.832197,53.644394,NaN,1005.827896,4.005758,NaN,5.462917,NaN,NaN
std,17.386327,20.194248,6.908704,31.946541,NaN,37.199589,3.856600,NaN,3.371499,NaN,NaN
min,-25.000000,20.000000,0.000000,0.000000,NaN,800.120000,0.000000,NaN,0.000000,NaN,NaN
25%,4.000000,57.000000,5.000000,19.000000,NaN,994.800000,1.000000,NaN,3.000000,NaN,NaN
50%,21.000000,70.000000,9.000000,58.000000,NaN,1007.650000,3.000000,NaN,5.000000,NaN,NaN
75%,31.000000,84.000000,13.500000,82.000000,NaN,1016.772500,7.000000,NaN,7.500000,NaN,NaN


## 3. Check for missing values

Before building a machine learning model, it is important to see whether any columns contain missing values.  
This notebook handles missing values inside the pipeline instead of editing the dataset manually.


In [23]:
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_summary

Temperature             0
Humidity                0
Wind Speed              0
Precipitation (%)       0
Cloud Cover             0
Atmospheric Pressure    0
UV Index                0
Season                  0
Visibility (km)         0
Location                0
Weather Type            0
dtype: int64

## 4. Split Features and Target

This cell separates the input columns (**features**) from the output column (**target**).  
The target for this project is `Weather Type`, which is the label the model will learn to predict.

In [25]:
TARGET_COLUMN = "Weather Type"

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Expected target column '{TARGET_COLUMN}', but found: {list(df.columns)}")

X = df.drop(columns=[TARGET_COLUMN])

# Encode the target weather condition labels as numbers.
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df[TARGET_COLUMN])
target_mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))

print("Feature columns:", list(X.columns))
print("Target encoding:", target_mapping)

Feature columns: ['Temperature', 'Humidity', 'Wind Speed', 'Precipitation (%)', 'Cloud Cover', 'Atmospheric Pressure', 'UV Index', 'Season', 'Visibility (km)', 'Location']
Target encoding: {'Cloudy': np.int64(0), 'Rainy': np.int64(1), 'Snowy': np.int64(2), 'Sunny': np.int64(3)}
